# Bài 7.1 · Dữ liệu âm thanh: nghe và nhìn

**Câu hỏi của buổi này:** một đoạn âm thanh đi vào model dưới dạng gì? Ta đi từ tiếng thật (giọng đọc, chim hót, kèn, cá voi) tới dãy số, tới spectrogram, tới log-mel, và **nghe** từng bước bằng trình phát ngay trong notebook.

| Mục | Sẽ thấy và nghe |
|---|---|
| 1. Âm thanh là dãy số | bảng bốn đoạn ghi âm, waveform, phóng to tới từng mẫu |
| 2. Sample rate và aliasing | tiếng quét tần số và tiếng kèn khi giảm sample rate đúng và sai |
| 3. STFT | ba spectrogram với ba độ dài window |
| 4. Mel và log-mel | bộ lọc mel, log-mel của bốn âm thanh |
| 5. Đoạn im lặng và log | vì sao cần một mức sàn trước khi lấy log |
| 6. Augmentation | dời, đổi âm lượng, thêm nhiễu, che dải trên log-mel |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **Internet** trong *Session options*; CPU là đủ, chạy hết khoảng 1 phút. Bài giảng: [ai-canuc80k · bài 7.1](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/7.1).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import librosa, scipy
from scipy.signal import resample_poly, chirp
from IPython.display import Audio

np.random.seed(0)
table(pd.DataFrame([('librosa', librosa.__version__), ('scipy', scipy.__version__), ('numpy', np.__version__)],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

def play(y, sr, label):
    display(HTML(f'<div style="margin:6px 0 2px;font-weight:600">{label}</div>')); display(Audio(y, rate=sr))

def spec(ax, y, sr, n_fft=1024, hop=256, title='', fmax=None, mel=False):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop)) ** 2
    if mel:
        S = librosa.feature.melspectrogram(S=S, sr=sr, n_mels=80)
    D = librosa.power_to_db(S, ref=np.max, top_db=80)
    t_end = len(y) / sr
    extent = [0, t_end, 0, 80] if mel else [0, t_end, 0, sr / 2]
    ax.imshow(D, origin='lower', aspect='auto', cmap='magma', extent=extent, vmin=-80, vmax=0)
    ax.set_xlabel('giây'); ax.set_ylabel('dải mel' if mel else 'Hz'); ax.set_title(title, fontsize=11); ax.grid(False)
    if fmax and not mel: ax.set_ylim(0, fmax)
    return D

## 1. Âm thanh là dãy số

Micro đo áp suất không khí rất nhiều lần mỗi giây. **Sample rate** $f_s$ là số lần đo mỗi giây; một đoạn dài $T$ giây là một dãy $f_s \times T$ số thực, thường nằm trong $[-1, 1]$. Bốn đoạn ghi âm mẫu của thư viện librosa, mỗi đoạn lấy 4 giây đầu:

In [ ]:
sources = [('libri1', 'giọng đọc'), ('robin', 'chim hót'), ('trumpet', 'kèn trumpet'), ('humpback', 'cá voi lưng gù')]
clips = {}
with quiet():
    for key, name in sources:
        clips[name], SR = librosa.load(librosa.ex(key), sr=22050, duration=4.0)
table(pd.DataFrame([(n, SR, len(y), len(y) / SR, y.min(), y.max()) for n, y in clips.items()],
                   columns=['đoạn', 'sample rate (Hz)', 'số mẫu', 'giây', 'min', 'max']),
      'Bốn đoạn ghi âm', fmt={'giây': '{:.2f}', 'min': '{:+.3f}', 'max': '{:+.3f}'})
check(f'đoạn giọng đọc dài 4 giây có đúng {SR} × 4 = {SR * 4} số', len(clips['giọng đọc']) == SR * 4)
for n, y in clips.items(): play(y, SR, n)

In [ ]:
y = clips['giọng đọc']
t = np.arange(len(y)) / SR
a0 = int(np.argmax(np.abs(y))) - 55
fig, axes = plt.subplots(1, 2, figsize=(15, 3.6), gridspec_kw={'width_ratios': [2, 1]})
axes[0].plot(t, y, lw=.5, color=C['blue']); axes[0].axvspan(t[a0], t[a0 + 110], color=C['clay'], alpha=.3)
axes[0].set(xlabel='giây', ylabel='biên độ', title='Waveform giọng đọc: 4 giây = 88 200 số')
seg = slice(a0, a0 + 110)
axes[1].plot(t[seg] * 1000, y[seg], color=C['gray'], lw=1); axes[1].plot(t[seg] * 1000, y[seg], 'o', ms=3, color=C['clay'])
axes[1].set(xlabel='mili giây', title=f'Phóng to 5 ms (vùng đỏ): {seg.stop - seg.start} mẫu, mỗi chấm một số')
plt.tight_layout(); plt.show()
note(f'Hai mẫu liền nhau cách nhau 1/{SR} giây ≈ {1000 / SR:.3f} ms. Waveform cho biết âm lượng theo thời gian, nhưng nhìn nó rất khó biết đó là tiếng gì.')

## 2. Sample rate và aliasing

Với sample rate $f_s$, tần số cao nhất còn giữ được là **Nyquist** $f_s/2$. Thành phần cao hơn không biến mất mà **gập** xuống: tần số $f$ (giữa $f_s/2$ và $f_s$) hiện ra ở $f_s - f$. Thử với một tiếng quét tần số từ 0 lên 8 000 Hz ở 16 000 Hz, rồi giảm xuống 8 000 Hz theo hai cách:

- **bỏ mẫu**: giữ một mẫu trong hai (`x[::2]`);
- **resample có lọc**: `resample_poly` lọc bỏ phần trên Nyquist mới rồi mới giảm mẫu.

In [ ]:
fs = 16000
tt = np.arange(0, 2, 1 / fs)
sweep = 0.3 * chirp(tt, f0=0, f1=8000, t1=2, method='linear')
naive, filt = sweep[::2], resample_poly(sweep, 1, 2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
spec(axes[0], sweep, fs, 512, 128, 'gốc 16 000 Hz: quét 0 → 8 000 Hz')
spec(axes[1], naive, 8000, 256, 64, 'bỏ mẫu xuống 8 000 Hz: nửa sau gập ngược')
spec(axes[2], filt, 8000, 256, 64, 'resample có lọc: nửa sau bị cắt, không bịa')
for ax in axes: ax.set_ylim(0, 8000)
plt.tight_layout(); plt.show()

def dominant(x, sr, at):
    i = int(at * sr) - 512; f = np.abs(np.fft.rfft(x[i:i + 1024] * np.hanning(1024)))      # cửa sổ 1024 mẫu có tâm tại `at`
    return np.fft.rfftfreq(1024, 1 / sr)[f.argmax()]
d_naive = dominant(naive, 8000, 1.5)
rms = lambda x: np.sqrt(np.mean(x ** 2))
ratio = rms(filt[len(filt) // 2 + 400:]) / rms(filt[:len(filt) // 2 - 400])
check(f'giây 1.5 (thật ra 6 000 Hz): bản bỏ mẫu nghe ở {d_naive:.0f} Hz ≈ 8 000 − 6 000', abs(d_naive - 2000) < 100)
check(f'bản có lọc: nửa sau còn {ratio:.3f} lần biên độ nửa đầu', ratio < 0.1)
play(sweep, fs, 'gốc'); play(naive, 8000, 'bỏ mẫu (nghe tiếng đi lên rồi đi xuống)'); play(filt, 8000, 'resample có lọc (nửa sau im)')

Tiếng kèn thật có nhiều họa âm cao. Giảm từ 22 050 Hz xuống 4 410 Hz (Nyquist mới 2 205 Hz):

In [ ]:
tr = clips['kèn trumpet']
tr_naive, tr_filt = tr[::5], resample_poly(tr, 1, 5)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
spec(axes[0], tr, SR, 2048, 512, f'gốc {SR} Hz', fmax=SR / 2)
spec(axes[1], tr_naive, SR // 5, 512, 128, f'bỏ mẫu xuống {SR // 5} Hz')
spec(axes[2], tr_filt, SR // 5, 512, 128, f'resample có lọc xuống {SR // 5} Hz')
plt.tight_layout(); plt.show()
play(tr, SR, 'gốc'); play(tr_naive, SR // 5, 'bỏ mẫu: có tiếng rít lạ'); play(tr_filt, SR // 5, 'có lọc: đục hơn nhưng sạch')
note('Bản bỏ mẫu có những vệt không có trong bản gốc: họa âm trên 2 205 Hz đã gập xuống. Model nhận vào sẽ học cả những âm giả ấy.')

## 3. STFT: tần số nào xuất hiện ở từng lúc

STFT cắt waveform thành các **frame** dài $L$ mẫu (window), cách nhau $H$ mẫu (hop), nhân mỗi frame với một cửa sổ Hann rồi lấy FFT. Mỗi frame thành một cột tần số; xếp các cột lại là **spectrogram**.

Window ngắn thì thấy rõ lúc nào âm bắt đầu nhưng tần số nhòe; window dài thì ngược lại.

In [ ]:
sp = clips['giọng đọc'][:int(1.6 * SR)]
cfgs = [(128, 32), (512, 128), (2048, 512)]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
rows = []
for ax, (n_fft, hop) in zip(axes, cfgs):
    D = spec(ax, sp, SR, n_fft, hop, f'window {n_fft} mẫu = {1000 * n_fft / SR:.0f} ms', fmax=5000)
    rows.append((n_fft, 1000 * n_fft / SR, hop, D.shape[1], D.shape[0], SR / n_fft))
plt.tight_layout(); plt.show()
stft_t = pd.DataFrame(rows, columns=['window (mẫu)', 'window (ms)', 'hop (mẫu)', 'số frame', 'số bin tần số', 'bin cách nhau (Hz)'])
table(stft_t, f'STFT của {len(sp)} mẫu (1.6 giây)', fmt={'window (ms)': '{:.1f}', 'bin cách nhau (Hz)': '{:.1f}'})
check('số frame = 1 + ⌊N / hop⌋ (librosa đệm hai đầu, center=True)', all(r[3] == 1 + len(sp) // r[2] for r in rows))
check('số bin = window / 2 + 1', all(r[4] == r[0] // 2 + 1 for r in rows))

## 4. Mel và log-mel

Tai người phân biệt rõ 100 Hz với 200 Hz, nhưng 8 000 Hz với 8 100 Hz thì gần như không. **Thang mel** dồn nhiều dải hẹp ở tần số thấp, ít dải rộng ở tần số cao. Mỗi dải là một bộ lọc tam giác; cộng năng lượng trong mỗi tam giác được một **mel band**. Rồi lấy **log** (đơn vị dB) vì năng lượng âm thanh trải qua nhiều bậc độ lớn.

In [ ]:
fb = librosa.filters.mel(sr=SR, n_fft=1024, n_mels=40)
freqs = np.linspace(0, SR / 2, fb.shape[1])
fig, ax = plt.subplots(figsize=(12, 3))
for k in range(fb.shape[0]):
    ax.plot(freqs, fb[k] / fb[k].max(), color=ORDER[k % 4], lw=1)
ax.set(xlabel='Hz', ylabel='trọng số', title='40 bộ lọc mel: hẹp và dày ở tần số thấp, rộng và thưa ở tần số cao')
plt.tight_layout(); plt.show()
check(f'ma trận mel có shape {fb.shape} = (số band, số bin STFT)', fb.shape == (40, 513))

S = np.abs(librosa.stft(sp, n_fft=1024, hop_length=256)) ** 2
M = librosa.feature.melspectrogram(S=S, sr=SR, n_mels=80)
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].imshow(S, origin='lower', aspect='auto', cmap='magma', extent=[0, len(sp) / SR, 0, SR / 2]); axes[0].set_ylim(0, 5000)
axes[0].set(title='năng lượng STFT, chưa lấy log: gần như chỉ thấy vài chấm', xlabel='giây', ylabel='Hz'); axes[0].grid(False)
spec(axes[1], sp, SR, 1024, 256, 'log-mel 80 band (dB): thấy rõ từng âm tiết', mel=True)
plt.tight_layout(); plt.show()
table(pd.DataFrame([('waveform', f'{len(sp)}'), ('STFT, window 1024, hop 256', f'{S.shape[0]} × {S.shape[1]}'), ('mel 80 band', f'{M.shape[0]} × {M.shape[1]}')],
                   columns=['bước', 'shape']), 'Kích thước qua từng bước')
note(f'Giá trị năng lượng lớn nhất gấp <b>{S.max() / np.median(S):.0e}</b> lần trung vị: không lấy log thì gần như mọi ô trông như số 0.')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (n, yy) in zip(axes, clips.items()):
    spec(ax, yy, SR, 1024, 256, n, mel=True)
fig.suptitle('Log-mel của bốn âm thanh: mỗi loại để lại một "dấu vân tay" khác nhau', y=1.04)
plt.tight_layout(); plt.show()

## 5. Đoạn im lặng và log

$\log 0 = -\infty$. Một đoạn im lặng tuyệt đối (thường gặp khi đệm cho đủ độ dài) cho ra $-\infty$, rồi NaN trong loss. Cách chữa: cộng hoặc chặn bằng một **mức sàn** nhỏ trước khi lấy log.

In [ ]:
padded = np.concatenate([clips['chim hót'][:SR], np.zeros(SR)])
Mp = librosa.feature.melspectrogram(y=padded, sr=SR, n_fft=1024, hop_length=256, n_mels=80)
with np.errstate(divide='ignore'):
    raw_log = 10 * np.log10(Mp)
floor_log = 10 * np.log10(np.maximum(Mp, 1e-10))
table(pd.DataFrame([('10·log10(năng lượng)', np.isinf(raw_log).mean(), np.min(raw_log)), ('10·log10(max(năng lượng, 1e-10))', np.isinf(floor_log).mean(), floor_log.min())],
                   columns=['cách lấy log', 'tỉ lệ ô bằng −∞', 'giá trị nhỏ nhất']), '1 giây chim hót + 1 giây đệm số 0', fmt={'tỉ lệ ô bằng −∞': '{:.1%}', 'giá trị nhỏ nhất': '{:.1f}'})
check('có mức sàn thì không còn −∞', np.isfinite(floor_log).all())

## 6. Augmentation

Những biến đổi giữ nguyên nhãn ("vẫn là tiếng chim"): dời thời gian, đổi âm lượng, trộn nhiễu với tỉ số tín hiệu/nhiễu (SNR) cho trước; và **SpecAugment** che ngẫu nhiên vài dải tần, vài đoạn thời gian ngay trên log-mel.

In [ ]:
rng = np.random.default_rng(0)
x = clips['giọng đọc'][:int(2.5 * SR)]
shifted = np.roll(x, int(0.4 * SR))
quiet_x = x * 10 ** (-12 / 20)
noise = rng.normal(size=len(x)); noise *= np.sqrt(np.mean(x ** 2) / 10 ** (10 / 10) / np.mean(noise ** 2))
noisy = x + noise
snr = 10 * np.log10(np.mean(x ** 2) / np.mean(noise ** 2))

def logmel(v):
    return librosa.power_to_db(librosa.feature.melspectrogram(y=v, sr=SR, n_fft=1024, hop_length=256, n_mels=80), ref=1.0, top_db=None)
base = logmel(x)
masked = base.copy()
for _ in range(2):
    f0 = rng.integers(0, 70); masked[f0:f0 + 10] = base.mean()
    t0 = rng.integers(0, base.shape[1] - 20); masked[:, t0:t0 + 20] = base.mean()

views = [('gốc', base), ('dời 0.4 giây', logmel(shifted)), ('nhỏ đi 12 dB', logmel(quiet_x)), (f'thêm nhiễu, SNR {snr:.0f} dB', logmel(noisy)), ('SpecAugment: che 2 dải tần, 2 đoạn', masked)]
vmin, vmax = base.max() - 80, base.max()
fig, axes = plt.subplots(1, 5, figsize=(18, 3.4))
for ax, (n, D) in zip(axes, views):
    ax.imshow(D, origin='lower', aspect='auto', cmap='magma', vmin=vmin, vmax=vmax); ax.set_title(n, fontsize=10); ax.axis('off')
plt.tight_layout(); plt.show()
check(f'SNR đo được {snr:.2f} dB, đúng mức 10 dB đặt ra', abs(snr - 10) < 0.01)
gain_shift = np.median(logmel(quiet_x) - base)
check(f'nhỏ đi 12 dB thì log-mel dời đều {gain_shift:.1f} dB', abs(gain_shift + 12) < 0.1)
play(x, SR, 'gốc'); play(shifted, SR, 'dời 0.4 giây'); play(quiet_x, SR, 'nhỏ đi 12 dB'); play(noisy, SR, f'thêm nhiễu trắng, SNR {snr:.0f} dB')
note('Đổi âm lượng chỉ cộng một hằng số vào log-mel: chuẩn hóa từng đoạn (trừ mean) sẽ xóa luôn khác biệt này. Nhiễu thì lấp đầy những chỗ vốn tối.')

## 7. Tóm tắt

In [ ]:
summary = [('sample rate các đoạn mẫu', f'{SR} Hz'),
           ('quét 6 000 Hz, bỏ mẫu xuống 8 000 Hz', f'nghe thành {d_naive:.0f} Hz'),
           ('quét, resample có lọc: biên độ nửa sau / nửa đầu', f'{ratio:.3f}'),
           ('STFT window 512, hop 128 trên 1.6 giây', f"{int(stft_t.iloc[1]['số bin tần số'])} bin × {int(stft_t.iloc[1]['số frame'])} frame"),
           ('mel 80 band trên 1.6 giây', f'{M.shape[0]} × {M.shape[1]}'),
           ('log không mức sàn: ô bằng −∞', f'{np.isinf(raw_log).mean():.1%}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** đổi `n_mels=80` thành 20 rồi nhìn log-mel mờ đi; thay tiếng kèn ở mục 2 bằng tiếng chim (rất cao) rồi giảm xuống 8 000 Hz; tự ghi âm giọng mình, tải lên Kaggle và chạy lại mục 4.